# Notebook 02 — Modelo de Classificação de Categoria

## Objetivo

Treinar um modelo capaz de prever a categoria financeira de uma transação
(Alimentação, Transporte, Saúde, Moradia, Educação, Lazer, Serviços) a
partir apenas da sua descrição textual (ex: "Uber", "Netflix").

## Resumo da jornada de validação (histórico completo em `documentacao_tecnica.md`, Fase 5)

O processo de avaliação deste modelo passou por descobertas importantes,
que moldaram a metodologia final usada neste notebook:

1. Um primeiro teste com split aleatório por **transação** deu 100% de
   acurácia — falso positivo de qualidade, causado por vocabulário fechado
   (mesmo estabelecimento aparecendo em treino e teste).
2. Corrigido para split por **estabelecimento nunca visto**, a acurácia
   caiu para valores mais realistas, revelando a necessidade de um
   vocabulário mais rico e redundante (já resolvido no Notebook 01).
3. Testes de split único mostraram grande variação de resultado
   dependendo da sorte do sorteio — o que motivou a adoção de
   **validação cruzada com múltiplas rodadas** como metodologia final de
   avaliação.

## O que a célula de código abaixo executa, em ordem

1. **Carrega** o dataset sintético de transações (Notebook 01);
2. **Avalia com validação cruzada** (10 rodadas, cada uma reservando um
   conjunto diferente de estabelecimentos exclusivamente para teste) —
   essa etapa serve apenas para **estimar** a qualidade esperada do
   modelo de forma confiável, sem depender da sorte de um único sorteio;
3. **Treina o modelo final de produção** com 100% dos dados disponíveis
   — diferente da etapa anterior, aqui o objetivo é maximizar o
   aprendizado, não medir desempenho (isso já foi feito no passo 2);
4. **Serializa** os artefatos de produção (`vetorizador_tfidf.pkl`,
   `modelo_categoria_producao.pkl`, `codificador_categorias.pkl`) na
   pasta `modelos/` do laboratório.

## Resultado esperado (validado na versão de produção original)

Acurácia média de ~86,79% (desvio padrão ~5,33%) e F1-macro médio de
~86,26% — métrica considerada confiável por resultar da média de 10
avaliações independentes, não de um único teste isolado.

In [1]:
# ============================================================
# Modelo de Classificação de Categoria — Célula única
# ============================================================
"""
Fluxo completo: carrega o dataset, avalia com validação cruzada (10
rodadas, para estimar desempenho de forma confiável), treina o modelo
final com 100% dos dados, e serializa os artefatos de produção.
"""

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score


def obter_caminhos_laboratorio() -> dict:
    """Centraliza os caminhos de pastas da versão revisada (laboratório)."""
    base = '/content/drive/MyDrive/Hackathon_OCI_G9/HACKATHON G9 - TEAM 20/versao_revisada_final'
    caminhos = {'base': base, 'datasets': os.path.join(base, 'datasets'), 'modelos': os.path.join(base, 'modelos')}
    os.makedirs(caminhos['datasets'], exist_ok=True)
    os.makedirs(caminhos['modelos'], exist_ok=True)
    return caminhos


def avaliar_com_multiplos_splits(df: pd.DataFrame, n_rodadas: int = 10) -> pd.DataFrame:
    """Avalia o modelo em N sorteios diferentes de estabelecimentos treino/teste.

    Evita decisões baseadas na sorte de um único sorteio — cada rodada
    reserva um conjunto diferente de estabelecimentos exclusivamente
    para teste (nunca vistos no treino).
    """
    resultados = []
    for rodada in range(n_rodadas):
        estab_treino, estab_teste = train_test_split(
            df['descricao'].unique(), test_size=0.25, random_state=rodada
        )
        df_treino = df[df['descricao'].isin(estab_treino)]
        df_teste = df[df['descricao'].isin(estab_teste)]

        vetorizador = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
        X_treino_vet = vetorizador.fit_transform(df_treino['descricao'])
        X_teste_vet = vetorizador.transform(df_teste['descricao'])

        modelo = LogisticRegression(max_iter=1000, random_state=rodada)
        modelo.fit(X_treino_vet, df_treino['categoria_codificada'])
        y_previsto = modelo.predict(X_teste_vet)

        resultados.append({
            'rodada': rodada,
            'acuracia': accuracy_score(df_teste['categoria_codificada'], y_previsto),
            'f1_macro': f1_score(df_teste['categoria_codificada'], y_previsto, average='macro'),
        })
    return pd.DataFrame(resultados)


def treinar_e_salvar_modelo_final(df: pd.DataFrame, encoder: LabelEncoder, pasta_modelos: str, seed: int = 42) -> None:
    """Treina o modelo definitivo com 100% dos dados e serializa os artefatos de produção."""
    vetorizador = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
    X_vetorizado = vetorizador.fit_transform(df['descricao'])

    modelo = LogisticRegression(max_iter=1000, random_state=seed)
    modelo.fit(X_vetorizado, df['categoria_codificada'])

    joblib.dump(vetorizador, os.path.join(pasta_modelos, 'vetorizador_tfidf.pkl'))
    joblib.dump(modelo, os.path.join(pasta_modelos, 'modelo_categoria_producao.pkl'))
    joblib.dump(encoder, os.path.join(pasta_modelos, 'codificador_categorias.pkl'))

    print(f"\n✅ Modelo final treinado com {len(df)} transações (100% do dataset)")
    print("✅ Artefatos salvos: vetorizador_tfidf.pkl, modelo_categoria_producao.pkl, codificador_categorias.pkl")


# ------------------------------------------------------------
# EXECUÇÃO
# ------------------------------------------------------------
caminhos_lab = obter_caminhos_laboratorio()
df = pd.read_csv(os.path.join(caminhos_lab['datasets'], 'transacoes_sinteticas.csv'))

encoder_categorias = LabelEncoder()
df['categoria_codificada'] = encoder_categorias.fit_transform(df['categoria'])
print(f"✅ Dataset carregado: {len(df)} transações\n")

df_resultados_cv = avaliar_com_multiplos_splits(df)
print("📊 Validação cruzada (10 rodadas):")
print(df_resultados_cv)
print(f"\n✅ Acurácia média: {df_resultados_cv['acuracia'].mean():.4f} (desvio: {df_resultados_cv['acuracia'].std():.4f})")
print(f"✅ F1-macro médio: {df_resultados_cv['f1_macro'].mean():.4f} (desvio: {df_resultados_cv['f1_macro'].std():.4f})")

treinar_e_salvar_modelo_final(df, encoder_categorias, caminhos_lab['modelos'])

Mounted at /content/drive
✅ Dataset carregado: 10000 transações

📊 Validação cruzada (10 rodadas):
   rodada  acuracia  f1_macro
0       0  0.860849  0.870368
1       1  0.893575  0.908122
2       2  0.933770  0.941596
3       3  0.875610  0.860319
4       4  0.754637  0.708224
5       5  0.851852  0.870900
6       6  0.910282  0.889555
7       7  0.852840  0.843014
8       8  0.821361  0.815773
9       9  0.924606  0.918541

✅ Acurácia média: 0.8679 (desvio: 0.0533)
✅ F1-macro médio: 0.8626 (desvio: 0.0656)

✅ Modelo final treinado com 10000 transações (100% do dataset)
✅ Artefatos salvos: vetorizador_tfidf.pkl, modelo_categoria_producao.pkl, codificador_categorias.pkl
